# IA Generativa con Embeddings en Cassandra
Generación de logs sintéticos y almacenamiento con embeddings para análisis semántico.

In [ ]:
# pip install cassandra-driver faker sentence-transformers
from cassandra.cluster import Cluster
from faker import Faker
from sentence_transformers import SentenceTransformer
import numpy as np
import uuid

In [ ]:
# Conectar y preparar Cassandra
cluster = Cluster(['127.0.0.1'])
session = cluster.connect()

session.execute("""
CREATE KEYSPACE IF NOT EXISTS logs_ia WITH replication = {
    'class': 'SimpleStrategy', 'replication_factor': '1'
};
""")

session.set_keyspace('logs_ia')

session.execute("""
CREATE TABLE IF NOT EXISTS log_entries (
    id UUID PRIMARY KEY,
    message TEXT,
    embedding LIST<FLOAT>
);
""")

In [ ]:
# Insertar logs sintéticos con embeddings
faker = Faker()
model = SentenceTransformer('all-MiniLM-L6-v2')

def insertar_logs(n=5):
    for _ in range(n):
        message = faker.sentence()
        emb = model.encode(message).tolist()
        session.execute("""
            INSERT INTO log_entries (id, message, embedding)
            VALUES (%s, %s, %s)
        """, (uuid.uuid4(), message, emb))

insertar_logs(5)